# COMPASS multivariate models

Elastic-net Cox and XGBoost survival:cox, each in "both" (full feature set)
and "baseline" (androgen-axis-only) configurations, at every landmark.
Requires `01_preprocessing.ipynb` to have built the merged `profile_data`
inputs under `prediction_inputs_<arm>/` first. The available-case sensitivity
analysis compares baseline labs with baseline Gleason + somatic calls on the
matched Gleason-versus-labs and somatic-versus-labs cohorts at each treatment landmark.

In [ ]:
ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc")
COHORTS = (
    "all",
    "metastatic_adt",      # retrospective ADT-intent stratum
    "metastatic_llm",      # met_diagnosis LLM metastatic status
)
# Orthogonal to COHORTS: "none" keeps every patient, "pre_adt_castrate"
# drops those with a castrate testosterone (<50 ng/dL) before ADT start,
# who were presumably androgen-deprived elsewhere first.
EXCLUSIONS = ("none", "pre_adt_castrate")
OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.FORCE_RERUN = OVERWRITE
RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS)

## Run multivariate models

Elastic-net (both/baseline) and XGBoost (both/baseline) arms. Set
`OVERWRITE = True` in the configuration cell to refit and replace existing outputs. With `False`, completed tasks are skipped.

In [ ]:
for run in RUNS:
    cp.run_multivariate(run)

## Available-case Gleason/somatic sensitivity

Fit two separate matched comparisons: Gleason versus labs and somatic versus labs.
Each cohort preserves the landmark-specific split and ADT-relative follow-up. Gleason uses
the score closest to each landmark among values documented by then. It runs at 0, +90,
and +180 days; it is not comparable to the observation-indexed univariate analyses.

In [ ]:
for run in RUNS:
    cp.run_multivariate_available_case_sensitivity(run)

### Available-case sensitivity results

Held-out AUC(t), C-index, and Brier metrics for each source-specific matched cohort.

In [ ]:
import pandas as pd

sensitivity_rows = []
for run in RUNS:
    for analysis in ('gleason', 'somatic'):
        for landmark_day in run['landmarks']:
            for feature_set, model, filename in (
                ('labs', 'cox', 'cox_agg_multivariable_metrics.csv'),
                ('somatic-gleason', 'cox', 'cox_agg_multivariable_metrics.csv'),
                ('labs', 'xgboost', 'landmark_xgboost_metrics.csv'),
                ('somatic-gleason', 'xgboost', 'landmark_xgboost_metrics.csv'),
            ):
                path = run['output_dir'] / 'sensitivity_available_case' / analysis / feature_set / model / f'landmark_{landmark_day}' / 'both' / filename
                if path.exists():
                    frame = pd.read_csv(path)
                    frame['analysis'] = analysis
                    frame['feature_set'] = feature_set
                    frame['source_path'] = str(path)
                    sensitivity_rows.append(frame)
available_case_sensitivity_df = pd.concat(sensitivity_rows, ignore_index=True) if sensitivity_rows else pd.DataFrame()
available_case_sensitivity_df

## Summary tables

Per-run C-index / mean AUC(t) / integrated Brier for every (model, landmark,
config), then combined across runs.

In [ ]:
summary_dfs = {cp.run_key(run): cp.summarize_outputs(run) for run in RUNS}
for label, df in summary_dfs.items():
    print(f"=== {label} ===")
    display(df)

In [ ]:
import pandas as pd

combined_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_summary_df

## Held-out per-patient risk scores

Every multivariable fit writes one row per held-out patient next to its metrics
(`*_patient_risks.csv`): the risk score, the duration, and the event indicator.
The file can carry two scoring schemes, distinguished by its `dataset` column,
and they are **not interchangeable**:

| `dataset` | Covers | Scored by |
|---|---|---|
| `test` | the held-out test block | the final model, refit on all of train/val |
| `cv_oof` | **every patient in the cohort** | the one outer CV fold that excluded them |

`test` is always written. `cv_oof` appears only when the fit was run with
`--out-of-fold-risks`, which is what gives a held-out score to every patient
rather than the test block alone. It re-tunes hyperparameters inside each outer
fold, so it costs roughly `--oof-outer-folds` times a normal run; reusing the
ordinary tuning folds instead would be far cheaper but leaky, since those same
folds choose the winning penalizer and would then be scoring the patients that
chose it.

These feed the risk-stratified figures (`risk_score_stratified_figures.py`),
which select one scheme via `--dataset` and compare the score against Gleason,
stage, and the TP53/PTEN/RB1 trio. Render the two to separate output
directories: they come from different models, so pooling them into one KM panel
would mix two score scales.

The cell below reports coverage rather than refitting. `n_patients` is the test
block; `n_oof_patients`/`n_oof_scored` are the out-of-fold coverage and read 0
for runs without `--out-of-fold-risks`. **A task whose metrics exist but whose
risk file is `missing` was fit before the risk-score output existed**: the
resume logic keys on the metrics file, so it will keep being skipped. Set
`OVERWRITE = True` in the configuration cell and re-run the fitting cell above
to regenerate those.

In [ ]:
risk_dfs = {cp.run_key(run): cp.summarize_patient_risks(run) for run in RUNS}
for label, df in risk_dfs.items():
    print(f"=== {label} ===")
    display(df.drop(columns=["path"]))

combined_risk_df = pd.concat(risk_dfs.values(), ignore_index=True)
missing = combined_risk_df.loc[combined_risk_df["status"] == "missing"]
if missing.empty:
    n_ok = int((combined_risk_df["status"] == "ok").sum())
    print(f"All {n_ok} multivariable tasks wrote held-out risk scores.")
else:
    print(
        f"{len(missing)} of {len(combined_risk_df)} tasks have no risk file. "
        "These were fit before the risk-score output existed; set OVERWRITE = True "
        "and re-run the fitting cell to regenerate them:"
    )
    display(missing[["run", "model", "landmark", "config", "endpoint"]])

# Out-of-fold coverage, reported apart from the test block because the two come
# from different models. A task with n_oof_patients == 0 simply was not run with
# --out-of-fold-risks; only a shortfall BETWEEN n_oof_scored and n_oof_patients
# means an outer fold failed to fit and some patients went unscored.
scored = combined_risk_df.loc[combined_risk_df["status"] == "ok"]
with_oof = scored.loc[scored["n_oof_patients"] > 0]
if with_oof.empty:
    print(
        "\nNo task carries full-cohort out-of-fold scores. Re-run the fitting "
        "cell with --out-of-fold-risks to give every patient a held-out score "
        "instead of the test block alone."
    )
else:
    short = with_oof.loc[with_oof["n_oof_scored"] < with_oof["n_oof_patients"]]
    print(
        f"\n{len(with_oof)} of {len(scored)} completed tasks carry full-cohort "
        f"out-of-fold scores."
    )
    if not short.empty:
        print(
            f"{len(short)} of those have unscored patients -- an outer fold "
            f"failed to fit, so the figures would be drawn on a subset:"
        )
        display(
            short[["run", "model", "landmark", "config",
                   "n_oof_scored", "n_oof_patients"]]
        )